In [1]:
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
import json
from datetime import datetime, timezone, timedelta
import pickle
import os
from src.utils.ficheros import guardarExcel, guardarExcelMulti
from pathlib import Path

GRAYLOG_URL = "http://silog.solvan.comun.adif"
USER = "E942433"
PASSWORD = "Xiaohuanghua5"

# def search_logs(query, from_date, to_date, stream_id=None):
#     url = f"{GRAYLOG_URL}/api/search/universal/absolute"
    
#     params = {
#         "query": query,
#         "from": from_date,
#         "to": to_date,
#         "offset": 0,
#         "fields": "timestamp,source,message,level"
#     }
    
#     # Añadir filtro de stream si se especifica
#     if stream_id:
#         params["filter"] = f"streams:{stream_id}"
    
#     headers = {"Accept": "application/json"}
    
#     response = requests.get(
#         url,
#         params=params,
#         auth=HTTPBasicAuth(USER, PASSWORD),
#         headers=headers
#     )
    
#     response.raise_for_status()
#     return response.json()


# Ver streams disponibles
def get_streams():
    url = f"{GRAYLOG_URL}/api/streams"
    response = requests.get(
        url,
        auth=HTTPBasicAuth(USER, PASSWORD),
        headers={"Accept": "application/json"}
    )
    response.raise_for_status()
    streams = response.json()["streams"]
    for s in streams:
        print(f"ID: {s['id']} | Nombre: {s['title']}")
    return streams



In [2]:
import time
def graylog_get(url, params, headers, max_retries=5):
    for attempt in range(max_retries):
        try:
            r = requests.get(
                url, params=params,
                auth=HTTPBasicAuth(USER, PASSWORD),
                headers=headers, timeout=30
            )
            if r.status_code == 500:
                raise requests.exceptions.HTTPError("500", response=r)
            r.raise_for_status()
            return r.json()

        except requests.exceptions.HTTPError as e:
            if e.response is not None and e.response.status_code == 500:
                raise  # Propagar 500 sin reintentar — lo gestiona fetch_window
            raise

        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            wait = 2 ** attempt
            print(f"\n⏱️  Error red — reintento {attempt+1}/{max_retries} en {wait}s...")
            time.sleep(wait)

    raise RuntimeError(f"❌ Fallaron {max_retries} reintentos")


def fetch_window(query, from_str, to_str, stream_id, headers,
                 batch_size=5000, _depth=0):
    """
    Descarga una ventana de tiempo. Si encuentra error 500 por offset alto,
    divide la ventana en 2 mitades y las descarga recursivamente.
    Máximo 8 niveles de recursión (ventana mínima ~1s).
    """
    if _depth > 8:
        print(f"\n⛔ Ventana demasiado densa incluso dividida: {from_str} → {to_str}")
        return []

    messages = []
    offset    = 0

    while True:
        params = {
            "query": query, "from": from_str, "to": to_str,
            "limit": batch_size, "offset": offset,
            "fields": "timestamp,source,message,level,contentType"
        }
        if stream_id:
            params["filter"] = f"streams:{stream_id}"

        try:
            data  = graylog_get(f"{GRAYLOG_URL}/api/search/universal/absolute", params, headers)
            batch = data.get("messages", [])
            total = data.get("total_results", 0)
            messages.extend(batch)
            offset += len(batch)
            print(f"  {from_str} → {to_str} | {offset}/{total}", end="\r")
            if offset >= total or not batch:
                break

        except requests.exceptions.HTTPError as e:
            if e.response is not None and e.response.status_code == 500:
                # Demasiados resultados con offset alto → dividir ventana en 2
                t_from = datetime.fromisoformat(from_str.replace("Z", "+00:00"))
                t_to   = datetime.fromisoformat(to_str.replace("Z", "+00:00"))
                mid    = t_from + (t_to - t_from) / 2
                mid_str = mid.strftime("%Y-%m-%dT%H:%M:%S.000Z")

                print(f"\n✂️  Dividiendo [{from_str} → {to_str}] (offset={offset}, depth={_depth})")

                # Descartar mensajes parciales de esta ventana y rehacer por mitades
                left  = fetch_window(query, from_str, mid_str, stream_id, headers,
                                     batch_size, _depth + 1)
                right = fetch_window(query, mid_str, to_str, stream_id, headers,
                                     batch_size, _depth + 1)
                return messages[:offset - len(batch)] + left + right
            raise

    return messages


def get_total_results(query, from_str, to_str, stream_id, headers):
    params = {"query": query, "from": from_str, "to": to_str,
              "limit": 1, "offset": 0, "fields": "timestamp"}
    if stream_id:
        params["filter"] = f"streams:{stream_id}"
    data = graylog_get(f"{GRAYLOG_URL}/api/search/universal/absolute", params, headers)
    return data.get("total_results", 0)


def search_logs(query, from_date, to_date, stream_id=None, limit=None,
                max_per_window=5000, checkpoint_file="checkpoint.pkl"):
    headers      = {"Accept": "application/json"}
    all_messages = []
    range_start  = datetime.fromisoformat(from_date.replace("Z", "+00:00"))
    range_end    = datetime.fromisoformat(to_date.replace("Z", "+00:00"))
    total_seconds = (range_end - range_start).total_seconds()

    # Reanudar desde checkpoint
    resume_from = range_start
    if os.path.exists(checkpoint_file):
        print(f"♻️  Reanudando desde checkpoint...")
        with open(checkpoint_file, "rb") as f:
            ckpt = pickle.load(f)
        all_messages = ckpt["messages"]
        resume_from  = ckpt["last_window_end"]
        print(f"   {len(all_messages):,} msgs ya descargados, continuando desde {resume_from}\n")

    # Calcular ventana óptima
    print("🔍 Calculando total de mensajes...")
    total_global = get_total_results(
        query,
        range_start.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        range_end.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        stream_id, headers
    )
    print(f"📊 Total: {total_global:,}")

    if total_global == 0:
        return []

    density        = total_global / total_seconds
    window_seconds = max(int((max_per_window / density) * 0.85), 5)
    remaining      = (range_end - resume_from).total_seconds()
    print(f"⚙️  Ventana: {window_seconds}s | Estimadas: {int(remaining/window_seconds)+1}\n")

    current      = resume_from
    window_count = 0

    while current < range_end:
        window_end = min(current + timedelta(seconds=window_seconds), range_end)
        from_str   = current.strftime("%Y-%m-%dT%H:%M:%S.000Z")
        to_str     = window_end.strftime("%Y-%m-%dT%H:%M:%S.000Z")

        try:
            batch = fetch_window(query, from_str, to_str, stream_id, headers)
            all_messages.extend(batch)
            window_count += 1
            print(f"✅ {from_str} → {to_str} | +{len(batch):,} | Total: {len(all_messages):,}")

        except RuntimeError as e:
            with open(checkpoint_file, "wb") as f:
                pickle.dump({"messages": all_messages, "last_window_end": current}, f)
            print(f"\n💾 Guardado emergencia: {len(all_messages):,} msgs")
            raise e

        # Checkpoint cada 50 ventanas
        if window_count % 50 == 0:
            with open(checkpoint_file, "wb") as f:
                pickle.dump({"messages": all_messages, "last_window_end": window_end}, f)
            print(f"💾 Checkpoint: {len(all_messages):,} msgs")

        current = window_end

        if limit and len(all_messages) >= limit:
            all_messages = all_messages[:limit]
            print(f"\n🛑 Límite alcanzado: {limit:,} msgs")
            break

    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)

    print(f"\n✅ Descarga completa: {len(all_messages):,} mensajes")
    return all_messages


def search_logs_to_dataframe(query, from_date, to_date, stream_id=None,
                              limit=None, checkpoint_file="checkpoint.pkl"):
    messages = search_logs(query, from_date, to_date, stream_id, limit,
                           checkpoint_file=checkpoint_file)
    if not messages:
        return pd.DataFrame()
    rows = [msg.get("message", msg) for msg in messages]
    return pd.DataFrame(rows)

In [3]:
data = search_logs(
    query="contentType:(Block OR Signal OR TrackCircuit OR LevelCrossing) AND ctc:MAC",
    from_date="2026-03-09T00:00:00.000Z",
    to_date="2026-03-10T00:00:00.000Z",
    stream_id="68fb73bc6456d79315e70710",
    limit=None  )

🔍 Calculando total de mensajes...
📊 Total: 1,075,496
⚙️  Ventana: 341s | Estimadas: 254

✅ 2026-03-09T00:00:00.000Z → 2026-03-09T00:05:41.000Z | +464 | Total: 464
✅ 2026-03-09T00:05:41.000Z → 2026-03-09T00:11:22.000Z | +348 | Total: 812
✅ 2026-03-09T00:11:22.000Z → 2026-03-09T00:17:03.000Z | +218 | Total: 1,030
✅ 2026-03-09T00:17:03.000Z → 2026-03-09T00:22:44.000Z | +404 | Total: 1,434
✅ 2026-03-09T00:22:44.000Z → 2026-03-09T00:28:25.000Z | +469 | Total: 1,903
✅ 2026-03-09T00:28:25.000Z → 2026-03-09T00:34:06.000Z | +384 | Total: 2,287
✅ 2026-03-09T00:34:06.000Z → 2026-03-09T00:39:47.000Z | +387 | Total: 2,674
✅ 2026-03-09T00:39:47.000Z → 2026-03-09T00:45:28.000Z | +280 | Total: 2,954
✅ 2026-03-09T00:45:28.000Z → 2026-03-09T00:51:09.000Z | +285 | Total: 3,239
✅ 2026-03-09T00:51:09.000Z → 2026-03-09T00:56:50.000Z | +560 | Total: 3,799
✅ 2026-03-09T00:56:50.000Z → 2026-03-09T01:02:31.000Z | +309 | Total: 4,108
✅ 2026-03-09T01:02:31.000Z → 2026-03-09T01:08:12.000Z | +238 | Total: 4,346
✅ 2

In [4]:
lista_messages = [item['message']['message'] for item in data]

In [5]:
df = pd.json_normalize(lista_messages)

In [6]:
dict_list = [json.loads(x) for x in lista_messages]

In [7]:
df = pd.json_normalize(dict_list)

In [8]:
df['header.timestampMSG'] = pd.to_datetime(df['header.timestampMSG'], unit='ms')

In [9]:
df['header.timeStampCTC'] = pd.to_datetime(df['header.timeStampCTC'], unit='ms')

In [10]:
df[df["messageType.ChangeState.trackCircuit.element.name"] == "CVA21"]

,version,header.crc,header.ctc,header.timestampMSG,header.timeStampCTC,header.originSystem,header.sequenceNumber,header.ContentType,messageType.ChangeState.trackCircuit.element.ctc,messageType.ChangeState.trackCircuit.element.interlock,...,messageType.ChangeState.levelCrossing.element.ctc,messageType.ChangeState.levelCrossing.element.interlock,messageType.ChangeState.levelCrossing.element.name,messageType.ChangeState.levelCrossing.state.upToDate,messageType.ChangeState.levelCrossing.state.alarm,messageType.ChangeState.levelCrossing.state.barrier,messageType.ChangeState.levelCrossing.state.interlocked,messageType.ChangeState.levelCrossing.state.LCBlownLamps.lamp1,messageType.ChangeState.levelCrossing.state.LCBlownLamps.lamp2,messageType.ChangeState.levelCrossing.state.manual


In [11]:
df_filter = df[["header.ctc","header.ContentType"]].copy()

In [12]:
df_filter["element.interlock"] = (
    df["messageType.ChangeState.trackCircuit.element.interlock"]
        .combine_first(df["messageType.ChangeState.block.element.interlock"])
        .combine_first(df["messageType.ChangeState.signal.element.interlock"])
        .combine_first(df["messageType.ChangeState.levelCrossing.element.interlock"])
)

In [13]:
df_filter["element.name"] = (
    df["messageType.ChangeState.trackCircuit.element.name"]
        .combine_first(df["messageType.ChangeState.block.element.name"])
        .combine_first(df["messageType.ChangeState.signal.element.name"])
        .combine_first(df["messageType.ChangeState.levelCrossing.element.name"])
)

In [14]:
df_filter["element.type"] = (
    df["messageType.ChangeState.trackCircuit.element.type"]
        .combine_first(df["messageType.ChangeState.signal.element.type"])
        .combine_first(df["messageType.ChangeState.signal.element.type"])
        #.combine_first(df["messageType.ChangeState.levelCrossing.element.type"])
)

In [15]:
df_filter["header.ctc"].unique()

array(['MAC'], dtype=object)

In [16]:
df_filter["element.interlock"].unique()

array(['SD', 'NP', 'MC', 'XP', 'CX', 'VG', 'HU', 'AU', 'AZ', 'GJ', 'AQ',
       'PI', 'LG', 'MT', 'AH', 'TH', 'RP', 'TR', 'CZ', 'CA', 'MH', 'MY',
       'CM', 'DU', 'MG', 'UC', 'MQ', 'FR', 'HC', 'FU', 'LT', 'GO', 'LN',
       'MP', 'HO', 'DE', 'BT', 'PR', 'CG', 'CI', 'AE', 'ES', 'SU', 'OD',
       'FB', 'GA', 'MV', 'VT', 'VB', 'BP', 'ZZ', 'SA', 'RC', 'PT', 'PA',
       'PO', 'TZ', 'EE', 'XT', 'VC', 'VE', 'TA', 'VI', 'LK', 'MA', 'VL',
       'XV', 'VV', 'CE', 'TT', 'SF', 'LE', 'AC', 'B3', 'AO', 'AI', 'AL',
       'AF', 'AB', 'TB', 'SO', 'SJ', 'SI', 'SH', 'SG', 'PY', 'PW', 'PQ',
       'PD', 'OS', 'PC', 'NR', 'NG', 'MX', 'MS', 'BS', 'AV', 'XS', 'VN',
       'BU', 'GZ', 'MN', 'MK', 'LM', 'EM', 'CW', 'LA', 'LU', 'CV', 'DO',
       'GI', 'JA', 'HH', 'EH', 'MI'], dtype=object)

In [127]:
MAC = df_filter[df_filter["element.interlock"] == "XP"].copy()
# MAC = df_filter.copy()

In [128]:
MAC.drop_duplicates(keep = "first")

,header.ctc,header.ContentType,element.interlock,element.name,element.type
13,MAC,TrackCircuit,XP,A232,Switch
1119,MAC,TrackCircuit,XP,A227,Switch
2903,MAC,TrackCircuit,XP,A224,Switch
4364,MAC,TrackCircuit,XP,A225,Switch
4383,MAC,Signal,XP,S9/PA,In-Out
...,...,...,...,...,...
802880,MAC,TrackCircuit,XP,A233,TrackCircuit
997335,MAC,TrackCircuit,XP,A308,TrackCircuit
1033997,MAC,TrackCircuit,XP,A310,TrackCircuit
1034216,MAC,TrackCircuit,XP,A306,TrackCircuit


In [129]:
import requests
def getElments(interlock,ctc):

    url = "http://topo.rail.api.elcano.operaciones.adif/msetopo/download/filesInterlock"

    payload = {
        "interlock": interlock,
         "ctc": ctc,
        #  "interlockName": "CHAMARTIN"

    }
    response = requests.post(url, json=payload)

    if response.status_code == 200:
        content_type = response.headers.get("Content-Type", "")
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None
    data= response.json()
    config_string = data["topoResource"]["configFileJson"]
    config_object = json.loads(config_string)
    topo_info = {
    "id": config_object["id"],
    "IdCatalogo": config_object["version"]["mieCatalogue"]["id"],
    "VersionCatalogo": config_object["version"]["mieCatalogue"]["version"],
    "IdCTC": config_object["info"]["ctc"]["id"],
    "Mnemónico": config_object["info"]["interlocking"],
    "NombreEnclavamiento": config_object["info"]["name"],
    "Dependencias": [d["code"] for d in config_object["info"]["dependencies"]]
    }

    data_rows = []

    for e in config_object["viewCtc"]["elements"]:
        row = topo_info.copy() 
        row.update({
            "ElementoID": e["id"],
            "NombreElemento": e["name"],
            "TipoElemento": e["type"],
            "SubtipoElemento": e["subtype"],
            "NombreCircuito":e["trackCircuitName"],
    
        })
        data_rows.append(row)

    df_topo = pd.DataFrame(data_rows)
    return df_topo


    
        

In [130]:
# topo_info = {
#     "id": config_object["id"],
#     "IdCatalogo": config_object["version"]["mieCatalogue"]["id"],
#     "VersionCatalogo": config_object["version"]["mieCatalogue"]["version"],
#     "IdCTC": config_object["info"]["ctc"]["id"],
#     "Mnemónico": config_object["info"]["interlocking"],
#     "NombreEnclavamiento": config_object["info"]["name"],
#     "Dependencias": [d["code"] for d in config_object["info"]["dependencies"]]
# }

# data_rows = []

# for e in config_object["viewCtc"]["elements"]:
#     row = topo_info.copy() 
#     row.update({
#         "ElementoID": e["id"],
#         "NombreElemento": e["name"],
#         "TipoElemento": e["type"],
#         "SubtipoElemento": e["subtype"],
  
#     })
#     data_rows.append(row)

# df_topo = pd.DataFrame(data_rows)



In [131]:
MAC.head(5)

,header.ctc,header.ContentType,element.interlock,element.name,element.type
13,MAC,TrackCircuit,XP,A232,Switch
36,MAC,TrackCircuit,XP,A232,Switch
50,MAC,TrackCircuit,XP,A232,Switch
57,MAC,TrackCircuit,XP,A232,Switch
71,MAC,TrackCircuit,XP,A232,Switch


In [132]:
CTC = MAC["header.ctc"].unique()[0]

In [133]:
interlocking = MAC["element.interlock"].unique()

In [134]:
# dfs = []

# for interlock in interlocking:
#     try:
#         df_topo = getElments(interlock, CTC)

#         if df_topo is None or df_topo.empty:
#             print(f"{interlock} returned empty")

#         else:
#             dfs.append(df_topo)

#         time.sleep(1)

#     except Exception as e:
#         print(f"{interlock} failed: {e}")

# df_topos = pd.concat(dfs, ignore_index=True)

In [136]:
df_topo = getElments("XP","MAC")

In [137]:
def to_camel_case(s):
    if pd.isna(s):
        return ""
    parts = str(s).split()
    return parts[0].lower() + ''.join(p.capitalize() for p in parts[1:])

MAC["ElementoID"] = (
    MAC["header.ctc"] + "." +
    MAC["element.interlock"] + "." +
    MAC["header.ContentType"].apply(to_camel_case) + "." +
    MAC["element.name"]
)

In [139]:
Chamartín = MAC[MAC["element.interlock"] == "XP"].copy()

In [140]:
Chamartín.drop_duplicates(inplace=True)

In [141]:
compared = df_topo[~df_topo["ElementoID"].str.lower().isin(Chamartín["ElementoID"].str.lower())].copy()

In [142]:
compared_1 = compared[~compared["ElementoID"].str.contains("undefined")].copy()

In [143]:
compared_2 = compared_1[~compared_1["ElementoID"].str.contains("alarm")].copy()

In [144]:
compared_3 = compared_2[compared_2["SubtipoElemento"] != "trackCircuitNotSignalized"].copy().reset_index()

In [145]:

compared_3

,index,id,IdCatalogo,VersionCatalogo,IdCTC,Mnemónico,NombreEnclavamiento,Dependencias,ElementoID,NombreElemento,TipoElemento,SubtipoElemento,NombreCircuito
0,17,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.department.operationControl.MAC.XP,MAC.XP,operationControl,unknown,
1,166,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.trackCircuit.PS1BS,PS1BS,trackCircuit,trackCircuit,PS1BS
2,174,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.block.PS1BS,PS1BS,blockConvencional,unknown,
3,195,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.trackCircuit.37,37,trackCircuit,trackCircuit,37
4,278,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.trackCircuit.31N,31N,trackCircuit,trackCircuit,31N
5,292,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.trackCircuit.T366,T366,trackCircuit,trackCircuit,T366
6,298,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.trackCircuit.A441AV,A441AV,trackCircuit,trackCircuit,A441AV


In [146]:

temp = Chamartín[
    ~Chamartín["ElementoID"].str.lower().isin(df_topo["ElementoID"].str.lower())
]

compared_4 = temp[
    ~temp["element.name"].str.lower().isin(df_topo["NombreCircuito"].str.lower())
].copy().reset_index(drop=True)

In [147]:
compared_4

,header.ctc,header.ContentType,element.interlock,element.name,element.type,ElementoID
0,MAC,Signal,XP,S11,In-Out,MAC.XP.signal.S11
1,MAC,TrackCircuit,XP,VI1BS,TrackCircuit,MAC.XP.trackcircuit.VI1BS
2,MAC,TrackCircuit,XP,VI2BS,TrackCircuit,MAC.XP.trackcircuit.VI2BS
3,MAC,TrackCircuit,XP,TALLER,TrackCircuit,MAC.XP.trackcircuit.TALLER
4,MAC,TrackCircuit,XP,TALERD,TrackCircuit,MAC.XP.trackcircuit.TALERD
5,MAC,TrackCircuit,XP,VK2,TrackCircuit,MAC.XP.trackcircuit.VK2
6,MAC,TrackCircuit,XP,VK3,TrackCircuit,MAC.XP.trackcircuit.VK3
7,MAC,TrackCircuit,XP,VK4,TrackCircuit,MAC.XP.trackcircuit.VK4
8,MAC,TrackCircuit,XP,VK5,TrackCircuit,MAC.XP.trackcircuit.VK5
9,MAC,TrackCircuit,XP,305,TrackCircuit,MAC.XP.trackcircuit.305


In [148]:
df_topo

,id,IdCatalogo,VersionCatalogo,IdCTC,Mnemónico,NombreEnclavamiento,Dependencias,ElementoID,NombreElemento,TipoElemento,SubtipoElemento,NombreCircuito
0,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.trackCircuit.304,304,trackCircuit,trackCircuit,304
1,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.trackCircuit.FOSO,FOSO,trackCircuit,trackCircuit,FOSO
2,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.trackCircuit.35,35,trackCircuit,trackCircuit,35
3,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.trackCircuit.A118,A118,switch,exhaust,A118
4,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.trackCircuit.T575B,T575B,switch,obtuseCrossingExhaust,A393
...,...,...,...,...,...,...,...,...,...,...,...,...
314,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.signal.E5,E5,signal,main,
315,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.block.OD4BS,OD4BS,blockConvencional,unknown,
316,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.trackCircuit.A113,A113,switch,switch,A113
317,MAC.XP,CHA1,20250111,MAC,XP,XP-Vicalvaro_Mercancias,[98201],MAC.XP.signal.405,405,signal,main,


In [149]:
data ={
    "Elementos_topo_sin_recibir":compared_3,
    "Elementos_recibido_sin_topo": compared_4
}

In [150]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\2026_03_09_V_MERCANCIAS.xlsx")

In [152]:
guardarExcelMulti(data,fname)

Guardando: Elementos_topo_sin_recibir (1/2)
Guardando: Elementos_recibido_sin_topo (2/2)
